# Interactive Efficient Frontier Dashboard

Module: Modern Portfolio Theory

## Lesson summary

This dashboard lets students change expected returns, correlations, and the risk-free rate to see how the efficient frontier, global minimum variance portfolio, and tangency portfolio respond.

## Learning objectives

By the end of this dashboard, students should be able to:

- connect expected returns and covariance to frontier geometry;
- identify the global minimum variance portfolio;
- interpret the tangency portfolio as the highest Sharpe risky allocation;
- explain why correlation assumptions change diversification benefits;
- distinguish analytical intuition from implementable allocation constraints.

## Frontier equations

For weights $w$, expected returns $\mu$, covariance matrix $\Sigma$, and risk-free rate $r_f$:

$$
\mu_p=w^\top\mu,\qquad \sigma_p=\sqrt{w^\top\Sigma w}.
$$

The tangency portfolio maximizes the Sharpe ratio:

$$
\max_w \frac{w^\top\mu-r_f}{\sqrt{w^\top\Sigma w}}
\quad\text{subject to}\quad
\mathbf{1}^\top w=1.
$$

The dashboard changes $\mu$, correlations, and $r_f$ to show how these equations reshape the frontier.

## Setup

In [ ]:
import os

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display
from ipywidgets import FloatSlider, interact

from src.portfolio_optimization import (
    efficient_frontier_variance,
    global_minimum_variance_weights,
    portfolio_return,
    portfolio_volatility,
    tangency_weights,
)

RUN_INTERACTIVE_WIDGETS = os.getenv("RUN_INTERACTIVE_WIDGETS", "1") == "1"

## Frontier inputs

In [ ]:
assets = ["mexican_equity", "global_equity", "mxn_bond", "inflation_linked_bond"]
base_expected_returns = pd.Series(
    [0.12, 0.10, 0.07, 0.085],
    index=assets,
    name="expected_return",
)
base_volatility = pd.Series([0.22, 0.17, 0.025, 0.075], index=assets, name="volatility")

In [ ]:
def covariance_from_assumptions(equity_correlation=0.55, bond_correlation=0.25, equity_bond_correlation=0.15):
    correlation = pd.DataFrame(
        [
            [1.00, equity_correlation, equity_bond_correlation, equity_bond_correlation],
            [equity_correlation, 1.00, equity_bond_correlation, equity_bond_correlation],
            [equity_bond_correlation, equity_bond_correlation, 1.00, bond_correlation],
            [equity_bond_correlation, equity_bond_correlation, bond_correlation, 1.00],
        ],
        index=assets,
        columns=assets,
    )
    return correlation.mul(base_volatility, axis=0).mul(base_volatility, axis=1)

## Interactive dashboard

Run this cell in JupyterLab with `uv run jupyter lab`.

In [ ]:
def plot_efficient_frontier_dashboard(
    mexican_equity_return=0.12,
    global_equity_return=0.10,
    risk_free_rate=0.065,
    equity_correlation=0.55,
    bond_correlation=0.25,
    equity_bond_correlation=0.15,
):
    expected_returns = base_expected_returns.copy()
    expected_returns["mexican_equity"] = mexican_equity_return
    expected_returns["global_equity"] = global_equity_return
    covariance = covariance_from_assumptions(
        equity_correlation=equity_correlation,
        bond_correlation=bond_correlation,
        equity_bond_correlation=equity_bond_correlation,
    )

    target_returns = np.linspace(expected_returns.min() * 0.9, expected_returns.max() * 1.1, 80)
    frontier_volatility = np.sqrt(
        efficient_frontier_variance(target_returns, expected_returns, covariance)
    )
    gmvp = global_minimum_variance_weights(covariance)
    tangency = tangency_weights(expected_returns, covariance, risk_free_rate=risk_free_rate)

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=frontier_volatility,
            y=target_returns,
            mode="lines",
            name="Efficient frontier",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=[portfolio_volatility(gmvp, covariance)],
            y=[portfolio_return(gmvp, expected_returns)],
            mode="markers",
            marker={"size": 11},
            name="GMVP",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=[portfolio_volatility(tangency, covariance)],
            y=[portfolio_return(tangency, expected_returns)],
            mode="markers",
            marker={"size": 11},
            name="Tangency",
        )
    )
    fig.update_layout(
        title=(
            "Efficient frontier dashboard | "
            f"GMVP volatility {portfolio_volatility(gmvp, covariance):.2%} | "
            f"Tangency Sharpe {(portfolio_return(tangency, expected_returns) - risk_free_rate) / portfolio_volatility(tangency, covariance):.2f}"
        ),
        xaxis_title="Annualized volatility",
        yaxis_title="Expected return",
        template="plotly_white",
        height=560,
    )
    fig.update_xaxes(tickformat=".1%")
    fig.update_yaxes(tickformat=".1%")
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))
    display(pd.DataFrame({"gmvp": gmvp, "tangency": tangency}))


if RUN_INTERACTIVE_WIDGETS:
    interact(
        plot_efficient_frontier_dashboard,
        mexican_equity_return=FloatSlider(value=0.12, min=0.04, max=0.22, step=0.005, readout_format=".3f"),
        global_equity_return=FloatSlider(value=0.10, min=0.04, max=0.20, step=0.005, readout_format=".3f"),
        risk_free_rate=FloatSlider(value=0.065, min=0.00, max=0.14, step=0.005, readout_format=".3f"),
        equity_correlation=FloatSlider(value=0.55, min=-0.20, max=0.95, step=0.05, readout_format=".2f"),
        bond_correlation=FloatSlider(value=0.25, min=-0.20, max=0.95, step=0.05, readout_format=".2f"),
        equity_bond_correlation=FloatSlider(value=0.15, min=-0.40, max=0.80, step=0.05, readout_format=".2f"),
    );
else:
    plot_efficient_frontier_dashboard()

## Model limitations

- The dashboard shows sensitivity to selected inputs; it is not an optimizer with transaction costs, taxes, or mandates.
- Small changes in expected returns can move the tangency portfolio sharply.
- Correlation scenarios are stylized and may not capture crisis-period dependence.